<a href="https://colab.research.google.com/github/ashrafsohail42003/Training-project/blob/main/NLP_Chapter3_AshrafAlkahlout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Name : Ashraf Sohail Alkahlout
#120250206
#Prof :Aiman Ahmed Abusamra

## Setup

In [1]:
import re, math, random
from collections import Counter, defaultdict
from fractions import Fraction

random.seed(42)

## Exercise 3.8 — Unsmoothed unigrams and bigrams

### Approach
Tokenisation is one sentence per line, lower-cased, alphabetic+apostrophe runs (so `i'll` is a single token). Each sentence is wrapped with `<s>` and `</s>` per the convention used for the *I am Sam* example on page 4.

The bigram MLE is computed via Eq. 3.11 directly:

$$P(w_n \mid w_{n-1}) \;=\; \frac{C(w_{n-1}\,w_n)}{C(w_{n-1})}.$$

I keep `<s>` in the unigram counts so that `p_bigram(w, "<s>", …)` is well-defined. `<s>` only ever appears as a *context*, never as a predicted word, so this does not bias the conditional estimate.

In [2]:
def tokenize(text):
    sents = []
    for line in text.strip().splitlines():
        toks = re.findall(r"[a-z']+", line.lower())
        if toks:
            sents.append(["<s>"] + toks + ["</s>"])
    return sents

def train(sents):
    uni = Counter()
    bi  = defaultdict(Counter)
    for s in sents:
        for i, w in enumerate(s):
            uni[w] += 1
            if i > 0:
                bi[s[i-1]][w] += 1
    N = sum(uni.values())
    return uni, bi, N

def p_unigram(w, uni, N):
    return uni[w] / N if N else 0.0

def p_bigram(w, prev, uni, bi):
    return (bi[prev][w] / uni[prev]) if uni[prev] else 0.0

### Verification on the *I am Sam* corpus (page 4)

The chapter prints six bigram probabilities from this three-sentence corpus. The implementation must reproduce them exactly.

In [3]:
sam = """I am Sam
Sam I am
I do not like green eggs and ham"""
sents_sam = tokenize(sam)
uni_s, bi_s, N_s = train(sents_sam)

checks = [("I","<s>",Fraction(2,3)), ("Sam","<s>",Fraction(1,3)),
          ("am","i",Fraction(2,3)), ("</s>","sam",Fraction(1,2)),
          ("sam","am",Fraction(1,2)), ("do","i",Fraction(1,3))]
for w, prev, expected in checks:
    got = p_bigram(w.lower(), prev.lower(), uni_s, bi_s)
    print(f"P({w}|{prev}) = {got:.4f}   expected {float(expected):.4f}")

P(I|<s>) = 0.6667   expected 0.6667
P(Sam|<s>) = 0.3333   expected 0.3333
P(am|i) = 0.6667   expected 0.6667
P(</s>|sam) = 0.5000   expected 0.5000
P(sam|am) = 0.5000   expected 0.5000
P(do|i) = 0.3333   expected 0.3333


All six match the chapter to four decimal places, so the trainer is correct. Note in particular `P(am|i) = 2/3` rather than the textually printed `2/3` for `P(am|I)` — case-folding makes both forms identical, which is the intended behaviour.

## Exercise 3.9 — Two corpora, compared

I use two small corpora that differ in *register* but are matched in size. Corpus A is conversational ("email-like" text); Corpus B is expository ("news-like" text about space missions). Both are 30 sentences. Same tokeniser, same `<s>`/`</s>` boundary tokens.

In [4]:
corpus_email = """hi sarah just checking in to see how the project is going
let me know if you need anything from my side
i'll be in the office tomorrow morning
thanks for sending the report yesterday it looks great
can you review the slides before friday
i think we should schedule a call next week
please let me know what time works for you
the team meeting is moved to wednesday
i forwarded the email to john for his input
sorry for the late reply i was out sick
hope you had a good weekend
i'll send the invoice by the end of the day
let me know if anything else comes up
thanks again for your help with this
i appreciate the quick turnaround on the design
just a quick reminder about the deadline tomorrow
i agree with your suggestions on the draft
can we move the meeting to a later time
i'll be working from home today
please find the attached document for review
let me know if you have any questions
i wanted to follow up on my last email
the client is happy with the latest version
thanks for the update appreciate it
i'll get back to you as soon as possible
hope this helps and let me know
please share the latest numbers when you get a chance
i'm looking forward to our discussion next week
thanks for your patience on this matter
let me know your thoughts whenever you have time"""

corpus_news = """the space station completed another orbit around the earth
researchers announced new findings about the solar system
the launch of the satellite has been delayed by two weeks
nasa confirmed the success of the latest mission
data from the telescope shows distant galaxies in detail
the rover continues to send images from the surface of mars
scientists are studying the composition of the lunar soil
the new mission will explore the moons of jupiter in depth
engineers are testing the propulsion system of the rocket
the agency plans a series of launches over the next year
recent observations suggest the presence of water on the asteroid
the spacecraft will perform a flyby of the dwarf planet
international partners contributed key components to the project
the mission control team monitored the descent very carefully
the orbit of the comet was calculated by the team
researchers published their results in a peer reviewed journal
the experiment will run for several months in low earth orbit
the new instrument provides higher resolution than previous designs
the launch window opens next month according to the schedule
analysis of the samples revealed unexpected mineral content
the crew completed a successful spacewalk outside the station
the project received additional funding from the government
the astronomers detected a faint signal from a distant source
the engineers solved the issue with the communication antenna
the spacecraft entered the atmosphere at very high speed
data collected during the flyby will be analyzed for years
the program represents a major step in space exploration
new propulsion technology could shorten future trips to mars
the research focuses on the long term effects of microgravity
the agency announced its plans for the next decade of missions"""

sents_email = tokenize(corpus_email)
sents_news  = tokenize(corpus_news)
uni_e, bi_e, Ne = train(sents_email)
uni_n, bi_n, Nn = train(sents_news)

print(f"Email   : sentences={len(sents_email)}, tokens={Ne}, vocab={len(uni_e)}")
print(f"News-ish: sentences={len(sents_news)}, tokens={Nn}, vocab={len(uni_n)}")

Email   : sentences=30, tokens=306, vocab=138
News-ish: sentences=30, tokens=344, vocab=173


In [5]:
def top_unigrams(uni, k=10):
    return [(w,c) for w,c in uni.most_common() if w not in ("<s>","</s>")][:k]

def top_bigrams(bi, k=10):
    flat = [((p,w),c) for p,row in bi.items() for w,c in row.items()
            if p not in ("<s>","</s>") and w not in ("<s>","</s>")]
    flat.sort(key=lambda x: -x[1])
    return flat[:k]

print("Top-10 unigrams (email vs news):")
te, tn = top_unigrams(uni_e), top_unigrams(uni_n)
for i in range(10):
    a = f"{te[i][0]:<10}{te[i][1]:>4}"
    b = f"{tn[i][0]:<14}{tn[i][1]:>4}"
    print(f"  {a}    |    {b}")

print()
print("Top-10 bigrams (email vs news):")
be, bn = top_bigrams(bi_e), top_bigrams(bi_n)
for i in range(10):
    a = f"{be[i][0][0]+' '+be[i][0][1]:<18}{be[i][1]:>3}"
    b = f"{bn[i][0][0]+' '+bn[i][0][1]:<22}{bn[i][1]:>3}"
    print(f"  {a}    |    {b}")

Top-10 unigrams (email vs news):
  the         20    |    the             49
  you          8    |    of              13
  for          8    |    a                7
  to           7    |    in               5
  let          6    |    new              4
  me           6    |    from             4
  know         6    |    to               4
  i            6    |    will             4
  a            5    |    orbit            3
  i'll         4    |    mission          3

Top-10 bigrams (email vs news):
  let me              6    |    of the                  7
  me know             6    |    from the                3
  know if             3    |    the launch              2
  thanks for          3    |    the new                 2
  the latest          2    |    the agency              2
  if you              2    |    the next                2
  you have            2    |    the spacecraft          2
  i'll be             2    |    the project             2
  for the             2    |  

### Observations

**Unigrams.** The function word `the` dominates both, but its dominance is much stronger in the news corpus (49/344 ≈ 14%) than in the email corpus (20/306 ≈ 7%). The email register elevates first- and second-person pronouns (`you`, `i`, `me`, `i'll`) into the top 10, none of which appear at the top of the news distribution. Conversely the news distribution is full of nominal-modification material (`of`, `new`, `mission`, `orbit`).

**Bigrams.** This is where the genre split is sharpest. The email model's most frequent bigrams are interactional templates: `let me`, `me know`, `know if`, `thanks for`, `i'll be`. The news model's most frequent bigrams are nominal compounds and prepositional patterns: `of the`, `from the`, `the launch`, `the spacecraft`, `the agency`. The same frequency rank is being filled by completely different lexical material — exactly the kind of behaviour the chapter alludes to in §3.5 when it warns that n-grams "do a better and better job of modeling the training corpus" but encode "specific facts about a given training corpus".

## Exercise 3.10 — Generate random sentences

Cumulative-distribution sampling on the bigram row of the previous word, exactly as described in §3.4: starting from `<s>`, sample the next token from $P(w_n \mid w_{n-1})$, append it, and repeat until `</s>` is drawn (or a length cap is hit).

In [6]:
def sample_next(prev, bi):
    row = bi[prev]
    total = sum(row.values())
    if total == 0:
        return "</s>"
    r = random.random() * total
    acc = 0
    for w, c in row.items():
        acc += c
        if acc >= r:
            return w
    return "</s>"

def generate(bi, max_len=25):
    out, prev = [], "<s>"
    for _ in range(max_len):
        w = sample_next(prev, bi)
        if w == "</s>":
            break
        out.append(w)
        prev = w
    return " ".join(out)

print("From email bigram model:")
random.seed(7)
for _ in range(5):
    print("  >", generate(bi_e))

print("\nFrom news bigram model:")
random.seed(7)
for _ in range(5):
    print("  >", generate(bi_n))

From email bigram model:
  > thanks for your help with your help with this helps and let me know if you have time
  > thanks again for your suggestions on the late reply i appreciate it looks great
  > let me know if you have any questions
  > please let me know your patience on this
  > thanks again for the project is happy with the late reply i appreciate the client is moved to see how the update appreciate the invoice

From news bigram model:
  > the latest mission
  > the rocket
  > the earth
  > the launch of water on the descent very high speed
  > the issue with the satellite has been delayed by the agency announced new findings about the next year


The samples look qualitatively similar to the bigram-Shakespeare samples in Fig. 3.4 of the chapter: locally fluent two- and three-word fragments, but no real long-range coherence. That is expected — the Markov assumption in Eq. 3.8 with $N=2$ has no memory beyond one word. Note also that the news-model samples are dominated by sentences starting with `the` (because `<s> → the` is by far the highest-probability transition under that model), which mirrors the unigram skew observed in Exercise 3.9.

## Exercise 3.11 — Perplexity on a test set

Following Eq. 3.17 (bigram) and Eq. 3.16 (unigram):

$$\mathrm{PP}_{\text{bi}}(W) = \sqrt[N]{\prod_{i=1}^{N} \frac{1}{P(w_i \mid w_{i-1})}}, \qquad
\mathrm{PP}_{\text{uni}}(W) = \sqrt[N]{\prod_{i=1}^{N} \frac{1}{P(w_i)}}.$$

Computed in log-space to avoid floating-point underflow:

$$\log \mathrm{PP}(W) = -\frac{1}{N} \sum_{i=1}^{N} \log P(w_i \mid \cdot).$$

In [7]:
def perplexity(test_sents, uni, bi, mode="bigram"):
    log_sum, N = 0.0, 0
    Nt = sum(uni.values())
    for s in test_sents:
        if mode == "unigram":
            for w in s[1:]:                 # exclude the <s> in position 0
                p = p_unigram(w, uni, Nt)
                if p == 0.0: return float("inf")
                log_sum += math.log(p); N += 1
        else:
            for i in range(1, len(s)):       # bigram (s[i-1], s[i])
                p = p_bigram(s[i], s[i-1], uni, bi)
                if p == 0.0: return float("inf")
                log_sum += math.log(p); N += 1
    return math.exp(-log_sum / N) if N else float("inf")

### Train/test split and evaluation

I split the email corpus 25/5 and retrain on the 25-sentence portion. Two evaluations are reported: perplexity on the *training* set (a sanity check, always finite under unsmoothed MLE because every training bigram has count ≥ 1) and perplexity on the held-out *test* set.

In [8]:
train_sents = sents_email[:25]
test_sents  = sents_email[25:]
uni_t, bi_t, _ = train(train_sents)
print(f"Train: {len(train_sents)} sents, Test: {len(test_sents)} sents")

print(f"PP(train) bigram  = {perplexity(train_sents, uni_t, bi_t, 'bigram'):.3f}")
print(f"PP(train) unigram = {perplexity(train_sents, uni_t, bi_t, 'unigram'):.3f}")
print(f"PP(test)  bigram  = {perplexity(test_sents,  uni_t, bi_t, 'bigram')}")
print(f"PP(test)  unigram = {perplexity(test_sents,  uni_t, bi_t, 'unigram')}")

Train: 25 sents, Test: 5 sents
PP(train) bigram  = 2.476
PP(train) unigram = 84.433
PP(test)  bigram  = inf
PP(test)  unigram = inf


### Diagnosis: why is held-out perplexity infinite?

Exercise 3.8 specified the *unsmoothed* MLE. Under Eq. 3.11, any bigram $(w_{i-1}, w_i)$ with $C(w_{i-1}\,w_i)=0$ in the training data has $P(w_i \mid w_{i-1}) = 0$, which collapses the entire product in Eq. 3.17 to zero and sends perplexity to $+\infty$. This is exactly what the chapter warns about on page 9 just before introducing smoothing. The unigram estimator has the same problem when the test set contains a word never seen in training.

Listing the offending bigrams confirms this:

In [9]:
unseen = set()
for s in test_sents:
    for i in range(1, len(s)):
        if bi_t[s[i-1]][s[i]] == 0:
            unseen.add((s[i-1], s[i]))
for prev, w in list(unseen)[:10]:
    print(f"  ({prev!r:>20}, {w!r:<14})  count = 0")
print(f"  ... {len(unseen)} unseen bigrams in total")

  (             'share', 'the'         )  count = 0
  (               'get', 'a'           )  count = 0
  (              'know', '</s>'        )  count = 0
  (            'please', 'share'       )  count = 0
  (               'and', 'let'         )  count = 0
  (                'to', 'our'         )  count = 0
  (          'patience', 'on'          )  count = 0
  (           'forward', 'to'          )  count = 0
  (                 'a', 'chance'      )  count = 0
  (              'this', 'matter'      )  count = 0
  ... 31 unseen bigrams in total


### Reading the numbers

The training-set numbers are interpretable on their own and confirm the chapter's qualitative claim that higher-order n-grams concentrate probability mass more sharply than lower-order ones. On the same 25-sentence training data the bigram model gives $\mathrm{PP} \approx 2.48$ — close to the lower bound of 1 — while the unigram model gives $\mathrm{PP} \approx 84.4$, much closer to the vocabulary size. The bigram model is conditioning on context and therefore far less surprised by the next word.

The held-out numbers reproduce the well-known pathology of unsmoothed MLE: 31 of the test bigrams were never observed in training, each one alone is sufficient to make $\mathrm{PP} = \infty$. Section 3.6 (add-1 / Laplace smoothing) is the chapter's resolution: replace $C(w_{n-1}w_n)/C(w_{n-1})$ with $(C(w_{n-1}w_n)+1) / (C(w_{n-1})+V)$, so no probability is ever exactly zero, and the geometric mean in Eq. 3.17 stays finite. Implementing that is outside the scope of 3.8–3.11 but is the natural next step.